In [154]:
import math
import pandas as pd


DB_PATH = "runs_index.csv"

DB = pd.read_csv(DB_PATH, index_col=0)


PROBLEMS = ['vanderpol', 'pollu', 'rober', 'orego', 'hires', 'davis-skodje']
PRETRAINING_OPTIONS = ['derivmatch', 'none']
TRAINING_OPTIONS = ['shooting', 'collocation', 'none']
SEED_OPTIONS = list(range(100, 150))  # seeds from 10 to 20 inclusive

DB.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4846 entries, run_00IK0gof to run_zzmXop3O
Data columns (total 48 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   E_VF_species_test           4268 non-null   object 
 1   E_VF_species_train          4268 non-null   object 
 2   E_VF_test                   3939 non-null   float64
 3   E_VF_train                  4248 non-null   float64
 4   E_spec_species_test         4268 non-null   object 
 5   E_spec_species_train        4268 non-null   object 
 6   E_spec_test                 3939 non-null   float64
 7   E_spec_train                4248 non-null   float64
 8   E_trajectory_species_test   4268 non-null   object 
 9   E_trajectory_species_train  4268 non-null   object 
 10  E_trajectory_test           3939 non-null   float64
 11  E_trajectory_train          4248 non-null   float64
 12  collocation_time            4846 non-null   float64
 13  computer           

### Preprocess

In [155]:
# filter out all with E_vf_specites_train not being present, means of removing old runs. 
DB = DB[DB['E_VF_test'].notnull()]

# Remove rows where seed is not between 10 and 20 (inclusive)
DB = DB[(DB['seed'] >= 100) & (DB['seed'] <= 149)]

for problem in PROBLEMS: 
    for pretraining in PRETRAINING_OPTIONS:
        for training in TRAINING_OPTIONS:
            for seed in SEED_OPTIONS:
                # Filter the DataFrame for the current combination of problem, pretraining, training, and seed
                filtered_rows = DB[(DB['problem'] == problem) & 
                                   (DB['pretraining'] == pretraining) & 
                                   (DB['training'] == training) & 
                                   (DB['seed'] == seed)]
                
                # If there are more than two rows with the same combination, print the details
                if len(filtered_rows) > 2:

                    print(f"Problem: {problem}, Pretraining: {pretraining}, Training: {training}, Seed: {seed}, Count: {len(filtered_rows)}")

                # Exit all loops after finding the first duplicate for this combination

Problem: vanderpol, Pretraining: derivmatch, Training: shooting, Seed: 100, Count: 3
Problem: vanderpol, Pretraining: derivmatch, Training: shooting, Seed: 101, Count: 4
Problem: vanderpol, Pretraining: derivmatch, Training: shooting, Seed: 102, Count: 3
Problem: vanderpol, Pretraining: derivmatch, Training: shooting, Seed: 104, Count: 4
Problem: vanderpol, Pretraining: derivmatch, Training: shooting, Seed: 105, Count: 3
Problem: vanderpol, Pretraining: derivmatch, Training: shooting, Seed: 106, Count: 3
Problem: vanderpol, Pretraining: derivmatch, Training: shooting, Seed: 107, Count: 5
Problem: vanderpol, Pretraining: derivmatch, Training: shooting, Seed: 111, Count: 3
Problem: vanderpol, Pretraining: derivmatch, Training: shooting, Seed: 114, Count: 3
Problem: vanderpol, Pretraining: derivmatch, Training: shooting, Seed: 116, Count: 3
Problem: vanderpol, Pretraining: derivmatch, Training: shooting, Seed: 120, Count: 3
Problem: vanderpol, Pretraining: derivmatch, Training: shooting, 

In [156]:
problem = "vanderpol"
model = "GELU-scaled"
for problem in PROBLEMS:
    for model in DB["model"].unique():
        print(f"Problem: {problem}, Model: {model}, Unique n_params: {DB[(DB['problem'] == problem) & (DB['model'] == model)]['n_params'].unique()}")
    print()

Problem: vanderpol, Model: GELU-scaled, Unique n_params: [317]
Problem: vanderpol, Model: mlp, Unique n_params: [282]
Problem: vanderpol, Model: stiff, Unique n_params: [268]

Problem: pollu, Model: GELU-scaled, Unique n_params: [4940]
Problem: pollu, Model: mlp, Unique n_params: [4940]
Problem: pollu, Model: stiff, Unique n_params: [4744]

Problem: rober, Model: GELU-scaled, Unique n_params: [332]
Problem: rober, Model: mlp, Unique n_params: [348]
Problem: rober, Model: stiff, Unique n_params: [318]

Problem: orego, Model: GELU-scaled, Unique n_params: [516]
Problem: orego, Model: mlp, Unique n_params: [516]
Problem: orego, Model: stiff, Unique n_params: [486]

Problem: hires, Model: GELU-scaled, Unique n_params: [855]
Problem: hires, Model: mlp, Unique n_params: [827]
Problem: hires, Model: stiff, Unique n_params: [816]

Problem: davis-skodje, Model: GELU-scaled, Unique n_params: [317]
Problem: davis-skodje, Model: mlp, Unique n_params: [282]
Problem: davis-skodje, Model: stiff, Uniq

In [157]:
# Filter out so that when there are two sets of parameters for the same problem+model, we only keep the one with the bigger number of parameters.
max_n_params = DB.groupby(["problem", "model"])["n_params"].transform("max")
DB = DB[DB["n_params"] == max_n_params]

make a function that takes in 'db' and 'problem'

db has the columns E_trajectory_test E_spec_test E_VF_test  seed, n_params training model 
model has three different values, training has two different values.

I want to create a table with the following rows: 
trajectory spec VF (all these are scalars), and also seed (this will be a list of seed) and n_params (list of n_params). The columns will first be the first three different models with the one training, and then again the same three models with the other training. So the columns will be: model1_training1, model2_training1, model3_training1, model1_training2, model2_training2, model3_training2.

Per model/training combination, there are different seeds. When calculating the different values, first filter out the 'k' rows with the worst E_trajectory_test, then calculate the mean of the remaining rows for E_trajectory_test, E_spec_test, and E_VF_test. For seed and n_params, just return the list of seeds and n_params for the remaining rows.

the function should return a pandas DataFrame with the specified rows and columns and hopefully also print those values nicely. 

In [158]:
def mainablation_results(db, problem, k=4, model_order=None, training_order=None, agg="mean"):
    """Build a model x training summary table for a given problem.

    Per (model, training) combination, drops the `k` rows with the worst
    (highest) E_trajectory_test and the `k` rows with the best (lowest)
    E_trajectory_test, then aggregates E_trajectory_test, E_spec_test,
    E_VF_test over the remaining rows with `agg` ("mean", "median" or "min");
    seed and n_params are kept as lists of the remaining rows' values.
    """
    sub = db[db["problem"] == problem]

    if model_order is None:
        model_order = sorted(sub["model"].dropna().unique())
    if training_order is None:
        # "none" means no post-training step (pretraining-only run); exclude it
        # so the default is the two real training methods (collocation, shooting).
        training_order = sorted(t for t in sub["training"].dropna().unique() if t != "none")

    columns = [f"{model}_{training}" for training in training_order for model in model_order]
    row_names = ["trajectory", "spec", "VF", "seed", "n_params"]
    table = pd.DataFrame(index=row_names, columns=columns, dtype=object)

    for training in training_order:
        for model in model_order:
            col = f"{model}_{training}"
            grp = sub[(sub["model"] == model) & (sub["training"] == training)]
            grp = grp.sort_values("E_trajectory_test", ascending=True)  # best first
            kept = grp.iloc[k:len(grp) - k] if k > 0 else grp

            table.loc["trajectory", col] = f"{kept['E_trajectory_test'].agg(agg):.3e}"
            table.loc["spec", col] = f"{kept['E_spec_test'].agg(agg):.3e}"
            table.loc["VF", col] = f"{kept['E_VF_test'].agg(agg):.3e}"
            table.loc["seed", col] = list(kept["seed"])
            table.loc["n_params", col] = list(kept["n_params"])

    # print(f"Main ablation results for problem={problem} (dropped {k} best and {k} worst E_trajectory_test run(s) per column)")
    # print(table.to_string())

    return table

for p in PROBLEMS:
    print(f"Main ablation results for problem={p} (dropped {2} best and {2} worst E_trajectory_test run(s) per column)")
    display(mainablation_results(DB, p))

Main ablation results for problem=vanderpol (dropped 2 best and 2 worst E_trajectory_test run(s) per column)


,GELU-scaled_collocation,mlp_collocation,stiff_collocation,GELU-scaled_shooting,mlp_shooting,stiff_shooting
trajectory,5.401e+01,5.783e+03,2.634e+03,6.685e+01,8.632e+03,1.259e+03
spec,6.390e-01,4.864e-01,1.013e-01,5.108e+01,4.010e-01,1.204e+01
VF,6.210e+01,1.762e+01,1.116e+00,5.105e+07,3.095e+01,6.560e+03
seed,"[104, 123, 121, 147, 118, 115, 149, 137, 130, ...","[134, 125, 127, 110, 121, 145, 108, 113, 126, ...","[103, 122, 123, 119, 118, 108, 124, 125, 142, ...","[122, 104, 121, 132, 118, 108, 126, 110, 149, ...","[107, 107, 129, 142, 121, 145, 126, 116, 120, ...","[101, 109, 133, 141, 111, 126, 110, 138, 105, ..."
n_params,"[317, 317, 317, 317, 317, 317, 317, 317, 317, ...","[282, 282, 282, 282, 282, 282, 282, 282, 282, ...","[268, 268, 268, 268, 268, 268, 268, 268, 268, ...","[317, 317, 317, 317, 317, 317, 317, 317, 317, ...","[282, 282, 282, 282, 282, 282, 282, 282, 282, ...","[268, 268, 268, 268, 268, 268, 268, 268, 268, ..."


Main ablation results for problem=pollu (dropped 2 best and 2 worst E_trajectory_test run(s) per column)


,GELU-scaled_collocation,mlp_collocation,stiff_collocation
trajectory,4.760e+02,1.713e+37,4.503e+00
spec,1.331e+273,inf,2.415e+302
VF,9.604e+03,4.748e+36,4.273e+08
seed,"[131, 102, 141, 109, 136, 106, 144, 115, 133, ...","[119, 111, 108, 107, 112, 135, 132, 123, 103, ...","[113, 107, 135, 145, 103, 133, 120, 138, 112, ..."
n_params,"[4940, 4940, 4940, 4940, 4940, 4940, 4940, 494...","[4940, 4940, 4940, 4940, 4940, 4940, 4940, 494...","[4744, 4744, 4744, 4744, 4744, 4744, 4744, 474..."


Main ablation results for problem=rober (dropped 2 best and 2 worst E_trajectory_test run(s) per column)


...

,GELU-scaled_collocation,mlp_collocation,stiff_collocation,GELU-scaled_shooting,mlp_shooting,stiff_shooting
trajectory,3.901e+02,1.371e+02,1.413e+00,1.678e+00,1.207e+00,1.114e+00
spec,6.541e+01,1.562e+01,9.061e-01,7.756e+01,1.550e+01,4.368e+00
VF,1.874e+00,8.145e-01,2.818e-04,3.653e+04,8.298e-01,1.735e+04
seed,"[138, 129, 127, 140, 148, 103, 107, 100, 112, ...","[126, 133, 147, 118, 149, 138, 112, 100, 146, ...","[112, 117, 100, 128, 138, 126, 116, 118, 141, ...","[101, 129, 143, 135, 138, 116, 109, 109, 127, ...","[138, 139, 108, 119, 147, 132, 109, 109, 120, ...","[101, 139, 100, 126, 133, 137, 112, 146, 132, ..."
n_params,"[516, 516, 516, 516, 516, 516, 516, 516, 516, ...","[516, 516, 516, 516, 516, 516, 516, 516, 516, ...","[486, 486, 486, 486, 486, 486, 486, 486, 486, ...","[516, 516, 516, 516, 516, 516, 516, 516, 516, ...","[516, 516, 516, 516, 516, 516, 516, 516, 516, ...","[486, 486, 486, 486, 486, 486, 486, 486, 486, ..."


Main ablation results for problem=hires (dropped 2 best and 2 worst E_trajectory_test run(s) per column)


,GELU-scaled_collocation,mlp_collocation,stiff_collocation,GELU-scaled_shooting,mlp_shooting,stiff_shooting
trajectory,2.222e+16,1.888e+09,2.969e+05,3.150e+05,1.688e+04,8.771e+04
spec,2.626e+14,1.168e+28,4.852e+28,4.035e+15,1.278e+31,1.063e+28
VF,4.577e+06,1.529e+08,5.422e+08,3.365e+07,1.572e+11,1.708e+09
seed,"[118, 144, 136, 143, 116, 145, 127, 147, 107, ...","[116, 145, 106, 119, 143, 140, 100, 125, 144, ...","[146, 131, 124, 114, 129, 135, 133, 149, 139, ...","[100, 101, 119, 116, 127, 114, 138, 147, 140, ...","[147, 104, 106, 144, 128, 113, 110, 101, 125, ...","[114, 115, 143, 136, 146, 137, 109, 125, 133, ..."
n_params,"[855, 855, 855, 855, 855, 855, 855, 855, 855, ...","[827, 827, 827, 827, 827, 827, 827, 827, 827, ...","[816, 816, 816, 816, 816, 816, 816, 816, 816, ...","[855, 855, 855, 855, 855, 855, 855, 855, 855, ...","[827, 827, 827, 827, 827, 827, 827, 827, 827, ...","[816, 816, 816, 816, 816, 816, 816, 816, 816, ..."


Main ablation results for problem=davis-skodje (dropped 2 best and 2 worst E_trajectory_test run(s) per column)


,GELU-scaled_collocation,mlp_collocation,stiff_collocation,GELU-scaled_shooting,mlp_shooting,stiff_shooting
trajectory,6.146e+09,1.410e+04,4.181e+12,4.606e+09,1.566e+08,1.142e+08
spec,9.261e-01,4.997e+00,5.064e-01,8.674e-01,2.793e+00,5.783e-01
VF,4.361e+06,2.204e+09,6.832e+05,8.215e+09,7.978e+12,2.924e+08
seed,"[140, 147, 109, 130, 111, 106, 101, 108, 132, ...","[122, 147, 103, 121, 129, 131, 144, 136, 140, ...","[135, 111, 127, 147, 141, 110, 100, 122, 101, ...","[138, 143, 130, 129, 108, 140, 114, 128, 101, ...","[110, 114, 133, 147, 149, 113, 138, 106, 104, ...","[137, 104, 148, 141, 120, 126, 116, 109, 119, ..."
n_params,"[317, 317, 317, 317, 317, 317, 317, 317, 317, ...","[282, 282, 282, 282, 282, 282, 282, 282, 282, ...","[268, 268, 268, 268, 268, 268, 268, 268, 268, ...","[317, 317, 317, 317, 317, 317, 317, 317, 317, ...","[282, 282, 282, 282, 282, 282, 282, 282, 282, ...","[268, 268, 268, 268, 268, 268, 268, 268, 268, ..."


In [159]:
def render_mainablation_row_group(table, problem_label="", model_order=("mlp", "GELU-scaled", "stiff"), training_order=("shooting", "collocation")):
    """Render the \\multirow Traj./Spec./VF body rows for one problem's
    mainablation_results table.

    The best (lowest) value in each row is bolded with \\textbf{}; missing or
    NaN values are rendered as "--".
    """
    columns = [f"{model}_{training}" for training in training_order for model in model_order]
    row_map = [("trajectory", "Traj."), ("spec", "Spec."), ("VF", "VF")]

    lines = [rf"\multirow{{{len(row_map)}}}{{*}}{{{problem_label}}}"]
    for row_key, row_label in row_map:
        cells, numeric = [], []
        for col in columns:
            raw = table.loc[row_key, col] if (row_key in table.index and col in table.columns) else None
            try:
                value = float(raw)
            except (TypeError, ValueError):
                value = math.nan
            if math.isnan(value):
                cells.append("--")
                numeric.append(math.inf)
            else:
                cells.append(str(raw))
                numeric.append(value)

        best_idx = numeric.index(min(numeric)) if numeric else None
        rendered = []
        for j, cell in enumerate(cells):
            if best_idx is not None and j == best_idx and cell != "--":
                cell = rf"\textbf{{{cell}}}"
            rendered.append(cell)

        lines.append(f"& {row_label} & " + " & ".join(rendered) + r" \\")

    return "\n".join(lines)


def render_mainablation_latex(table, problem_label="", label="tab:main-results",
                               model_order=("mlp", "GELU-scaled", "stiff"),
                               training_order=("shooting", "collocation")):
    """Take the output of mainablation_results for one problem and render the
    full LaTeX table (header + Traj./Spec./VF rows) with the numbers filled in.
    """
    header = [
        rf"\label{{{label}}}",
        r"\resizebox{\textwidth}{!}{%",
        r"\begin{tabular}{ll|ccc|ccc}",
        r"\toprule",
        r"& & \multicolumn{3}{c|}{\textbf{Single Shooting}} & \multicolumn{3}{c}{\textbf{Collocation}} \\",
        r"\cmidrule(lr){3-5} \cmidrule(lr){6-8}",
        r"\textbf{Problem} & \textbf{Metric}",
        r"& \textbf{MLP}",
        r"& \textbf{GELU}",
        r"& \textbf{StiffNet}",
        r"& \textbf{MLP}",
        r"& \textbf{GELU}",
        r"& \textbf{StiffNet} \\",
        r"\midrule",
    ]
    body = render_mainablation_row_group(table, problem_label, model_order, training_order)
    footer = [r"\bottomrule", r"\end{tabular}", r"}"]
    return "\n".join(header + [body] + footer)


print(render_mainablation_latex(mainablation_results(DB, "vanderpol"), problem_label="Van der Pol"))

\label{tab:main-results}
\resizebox{\textwidth}{!}{%
\begin{tabular}{ll|ccc|ccc}
\toprule
& & \multicolumn{3}{c|}{\textbf{Single Shooting}} & \multicolumn{3}{c}{\textbf{Collocation}} \\
\cmidrule(lr){3-5} \cmidrule(lr){6-8}
\textbf{Problem} & \textbf{Metric}
& \textbf{MLP}
& \textbf{GELU}
& \textbf{StiffNet}
& \textbf{MLP}
& \textbf{GELU}
& \textbf{StiffNet} \\
\midrule
\multirow{3}{*}{Van der Pol}
& Traj. & 8.632e+03 & 6.685e+01 & 1.259e+03 & 5.783e+03 & \textbf{5.401e+01} & 2.634e+03 \\
& Spec. & 4.010e-01 & 5.108e+01 & 1.204e+01 & 4.864e-01 & 6.390e-01 & \textbf{1.013e-01} \\
& VF & 3.095e+01 & 5.105e+07 & 6.560e+03 & 1.762e+01 & 6.210e+01 & \textbf{1.116e+00} \\
\bottomrule
\end{tabular}
}


In [160]:
PROBLEM_DISPLAY = {
    "vanderpol": "VDPOL",
    "pollu": "POLLU",
    "rober": "ROBER",
    "orego": "OREGO",
    "hires": "HIRES",
    "davis-skodje": "Davis--Skodje",
}

def render_mainablation_latex_all(db, problems=PROBLEMS, k=2,
                                   model_order=("mlp", "GELU-scaled", "stiff"),
                                   training_order=("shooting", "collocation"),
                                   problem_display=PROBLEM_DISPLAY,
                                   label="tab:main-results", agg="mean"):
    """Loop mainablation_results over `problems` and stack their row-groups
    into the full LaTeX table, separated by \\midrule.
    """
    header = [
        rf"\label{{{label}}}",
        r"\resizebox{\textwidth}{!}{%",
        r"\begin{tabular}{ll|ccc|ccc}",
        r"\toprule",
        r"& & \multicolumn{3}{c|}{\textbf{Single Shooting}} & \multicolumn{3}{c}{\textbf{Collocation}} \\",
        r"\cmidrule(lr){3-5} \cmidrule(lr){6-8}",
        r"\textbf{Problem} & \textbf{Metric}",
        r"& \textbf{MLP}",
        r"& \textbf{GELU}",
        r"& \textbf{StiffNet}",
        r"& \textbf{MLP}",
        r"& \textbf{GELU}",
        r"& \textbf{StiffNet} \\",
        r"\midrule",
    ]

    body = []
    for problem in problems:
        table = mainablation_results(db, problem, k=k, model_order=list(model_order), training_order=list(training_order), agg=agg)
        label_text = problem_display.get(problem, problem)
        body.append(render_mainablation_row_group(table, label_text, model_order, training_order))
        body.append(r"\midrule")
    if body and body[-1] == r"\midrule":
        body.pop()  # no trailing midrule after the last problem

    footer = [r"\bottomrule", r"\end{tabular}", r"}"]
    return "\n".join(header + body + footer)


print(render_mainablation_latex_all(DB))

\label{tab:main-results}
\resizebox{\textwidth}{!}{%
\begin{tabular}{ll|ccc|ccc}
\toprule
& & \multicolumn{3}{c|}{\textbf{Single Shooting}} & \multicolumn{3}{c}{\textbf{Collocation}} \\
\cmidrule(lr){3-5} \cmidrule(lr){6-8}
\textbf{Problem} & \textbf{Metric}
& \textbf{MLP}
& \textbf{GELU}
& \textbf{StiffNet}
& \textbf{MLP}
& \textbf{GELU}
& \textbf{StiffNet} \\
\midrule
\multirow{3}{*}{VDPOL}
& Traj. & 9.313e+03 & \textbf{1.196e+02} & 6.640e+03 & 6.322e+03 & 1.269e+02 & 4.277e+03 \\
& Spec. & 4.049e-01 & 4.576e+01 & 1.087e+01 & 9.571e-01 & 6.402e-01 & \textbf{9.828e-02} \\
& VF & 3.481e+01 & 4.568e+07 & 5.928e+03 & 5.517e+01 & 6.153e+01 & \textbf{1.037e+00} \\
\midrule
\multirow{3}{*}{POLLU}
& Traj. & -- & -- & -- & 2.008e+37 & 6.470e+02 & \textbf{7.886e+00} \\
& Spec. & -- & -- & -- & inf & \textbf{1.215e+273} & 2.563e+302 \\
& VF & -- & -- & -- & 4.335e+36 & \textbf{9.900e+03} & 4.384e+08 \\
\midrule
\multirow{3}{*}{ROBER}
& Traj. & 1.482e+23 & 1.514e+23 & 2.687e+02 & 1.976e+32 & 4.1

### Median across seeds

In [161]:
print(render_mainablation_latex_all(DB, agg="median", label="tab:main-results-median"))

\label{tab:main-results-median}
\resizebox{\textwidth}{!}{%
\begin{tabular}{ll|ccc|ccc}
\toprule
& & \multicolumn{3}{c|}{\textbf{Single Shooting}} & \multicolumn{3}{c}{\textbf{Collocation}} \\
\cmidrule(lr){3-5} \cmidrule(lr){6-8}
\textbf{Problem} & \textbf{Metric}
& \textbf{MLP}
& \textbf{GELU}
& \textbf{StiffNet}
& \textbf{MLP}
& \textbf{GELU}
& \textbf{StiffNet} \\
\midrule
\multirow{3}{*}{VDPOL}
& Traj. & 7.064e+03 & 7.408e+00 & 4.509e+00 & 5.327e+03 & 3.105e+00 & \textbf{2.032e+00} \\
& Spec. & 3.853e-01 & 7.180e-01 & 2.031e-01 & 3.753e-01 & 4.950e-01 & \textbf{6.033e-02} \\
& VF & 6.858e+00 & 1.149e+03 & 6.558e+00 & 3.879e+00 & 6.510e+00 & \textbf{3.860e-01} \\
\midrule
\multirow{3}{*}{POLLU}
& Traj. & -- & -- & -- & 7.916e+36 & 1.272e+02 & \textbf{1.451e-02} \\
& Spec. & -- & -- & -- & 3.489e+305 & \textbf{8.740e+266} & 1.435e+302 \\
& VF & -- & -- & -- & 6.351e+28 & \textbf{2.303e+03} & 6.251e+07 \\
\midrule
\multirow{3}{*}{ROBER}
& Traj. & 4.082e+03 & 5.918e+20 & 1.146e+01 & 1

### Minimum across seeds

In [162]:
# k=0: keep all seeds, otherwise the k best runs (incl. the minimum) would be dropped
print(render_mainablation_latex_all(DB, k=0, agg="min", label="tab:main-results-min"))

\label{tab:main-results-min}
\resizebox{\textwidth}{!}{%
\begin{tabular}{ll|ccc|ccc}
\toprule
& & \multicolumn{3}{c|}{\textbf{Single Shooting}} & \multicolumn{3}{c}{\textbf{Collocation}} \\
\cmidrule(lr){3-5} \cmidrule(lr){6-8}
\textbf{Problem} & \textbf{Metric}
& \textbf{MLP}
& \textbf{GELU}
& \textbf{StiffNet}
& \textbf{MLP}
& \textbf{GELU}
& \textbf{StiffNet} \\
\midrule
\multirow{3}{*}{VDPOL}
& Traj. & 2.755e+00 & 1.058e+00 & 5.866e-01 & 1.063e+00 & 1.058e+00 & \textbf{4.306e-02} \\
& Spec. & 1.224e-01 & 1.801e-01 & 4.962e-02 & 1.292e-01 & 1.801e-01 & \textbf{5.881e-03} \\
& VF & 4.739e-01 & 2.532e-01 & 2.340e-01 & 3.970e-01 & 2.532e-01 & \textbf{5.065e-02} \\
\midrule
\multirow{3}{*}{POLLU}
& Traj. & -- & -- & -- & 2.652e+25 & 5.326e+00 & \textbf{1.803e-04} \\
& Spec. & -- & -- & -- & 6.316e+292 & \textbf{1.289e+265} & 1.203e+301 \\
& VF & -- & -- & -- & 1.780e+26 & \textbf{1.372e+02} & 3.572e+04 \\
\midrule
\multirow{3}{*}{ROBER}
& Traj. & 1.298e+00 & 3.817e+17 & 1.423e-01 & 1.62

In [163]:
DB["seed"].unique()
# DB["pretraining"].unique()
# DB["training"].unique()

array([111, 113, 108, 139, 129, 105, 127, 142, 148, 135, 136, 102, 138,
       145, 147, 149, 122, 114, 128, 123, 126, 119, 144, 101, 107, 143,
       103, 141, 100, 125, 106, 132, 133, 134, 137, 112, 146, 110, 116,
       117, 118, 124, 131, 130, 121, 109, 120, 104, 140, 115])